# SmartParkingGU - Ejecucion en Google Colab

Este notebook orquesta todo el pipeline:
1. Clonar repositorio e instalar dependencias
2. Descargar dataset desde Google Drive (carpetas A y B)
3. Preparar dataset (merge + split estratificado)
4. Entrenar modelo base (notebook 02)
5. Fine-tuning y optimizacion (notebook 03)
6. Descargar resultados

In [ ]:
# CELDA 1: Clonar repositorio
!git clone https://github.com/KaraIbr/SmartParkingGU.git
%cd SmartParkingGU

In [ ]:
# CELDA 2: Instalar dependencias
!pip install -r requirements.txt -q

In [ ]:
# CELDA 3: Verificar GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: Sin GPU. El entrenamiento sera muy lento.')
    print('Ve Runtime > Change runtime type > GPU')

## Descargar Dataset desde Google Drive

El dataset esta en: https://drive.google.com/drive/folders/1kEVCDNzEl3_X35A3yOHJn0H62cUiqAoK

Carpetas:
- **A/** y **B/**: Dos vistas del estacionamiento
- Cada una tiene **busy/** (ocupado) y **free/** (libre)

In [ ]:
# CELDA 4: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Verificar que las carpetas A y B existen en Drive
import os
DRIVE_PATH = '/content/drive/MyDrive/CNRPark-Patches-150x150'

if os.path.exists(DRIVE_PATH):
    print(f'Dataset encontrado en: {DRIVE_PATH}')
    for folder in sorted(os.listdir(DRIVE_PATH)):
        folder_path = os.path.join(DRIVE_PATH, folder)
        if os.path.isdir(folder_path):
            subfolders = [d for d in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, d))]
            print(f'  {folder}/ -> {subfolders}')
            for sub in subfolders:
                sub_path = os.path.join(folder_path, sub)
                count = len([f for f in os.listdir(sub_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
                print(f'    {sub}/ -> {count} imagenes')
else:
    print(f'ERROR: No se encontro el dataset en {DRIVE_PATH}')
    print('Verifica que tengas el dataset en tu Drive personal')

In [ ]:
# CELDA 5: Copiar dataset del Drive al workspace local
!mkdir -p dataset/raw
!cp -r "{DRIVE_PATH}/A" dataset/raw/A
!cp -r "{DRIVE_PATH}/B" dataset/raw/B

# Verificar
for folder in ['dataset/raw/A', 'dataset/raw/B']:
    for sub in ['busy', 'free']:
        path = os.path.join(folder, sub)
        count = len([f for f in os.listdir(path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        print(f'{path}: {count} imagenes')

## Preparar Dataset

Merge carpetas A + B, renombrar a `ocupado`/`no_ocupado`, y dividir 70/15/15 estratificado.

In [ ]:
# CELDA 6: Preparar dataset - merge A+B, rename, split estratificado
import shutil
import random
from collections import defaultdict

random.seed(42)

# Crear estructura destino
for split in ['entrenamiento', 'validacion', 'test']:
    for cls in ['ocupado', 'no_ocupado']:
        os.makedirs(f'dataset/{split}/{cls}', exist_ok=True)

# Mapeo: busy -> ocupado, free -> no_ocupado
CLASS_MAP = {'busy': 'ocupado', 'free': 'no_ocupado'}

# Recopilar imagenes de A y B
all_images = {'ocupado': [], 'no_ocupado': []}

for src_folder in ['dataset/raw/A', 'dataset/raw/B']:
    for src_class, dst_class in CLASS_MAP.items():
        src_path = os.path.join(src_folder, src_class)
        if os.path.exists(src_path):
            imgs = [f for f in os.listdir(src_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            for img in imgs:
                all_images[dst_class].append(os.path.join(src_path, img))

print('Distribucion del dataset:')
for cls, imgs in all_images.items():
    print(f'  {cls}: {len(imgs)} imagenes')
print(f'  Total: {sum(len(v) for v in all_images.values())} imagenes')

# Split estratificado: 70% train, 15% val, 15% test
SPLITS = {'entrenamiento': 0.70, 'validacion': 0.15, 'test': 0.15}

for cls, imgs in all_images.items():
    random.shuffle(imgs)
    n = len(imgs)
    n_train = int(n * SPLITS['entrenamiento'])
    n_val = int(n * SPLITS['validacion'])

    splits = {
        'entrenamiento': imgs[:n_train],
        'validacion': imgs[n_train:n_train + n_val],
        'test': imgs[n_train + n_val:],
    }

    for split_name, split_imgs in splits.items():
        dst_dir = f'dataset/{split_name}/{cls}'
        for img_path in split_imgs:
            shutil.copy2(img_path, os.path.join(dst_dir, os.path.basename(img_path)))

print('\nDataset preparado:')
for split in ['entrenamiento', 'validacion', 'test']:
    for cls in ['ocupado', 'no_ocupado']:
        path = f'dataset/{split}/{cls}'
        count = len(os.listdir(path))
        print(f'  {path}: {count} imagenes')

## Ejecutar Notebooks de Entrenamiento

In [ ]:
# CELDA 7: Ejecutar notebook 02 - Modelo Base
# Esto entrena la CNN desde cero y guarda metrics.json + best_model.pt
%run notebooks/02_base_model_cnrpark.ipynb

In [ ]:
# CELDA 8: Ejecutar notebook 03 - Fine-Tuning
# Grid Search + Random Search + modelo optimizado
%run notebooks/03_finetune_cnrpark.ipynb

## Ver Resultados

In [ ]:
# CELDA 9: Ver resultados del fine-tuning
import pandas as pd

RESULTS_CSV = 'experiments/cnrpark/results.csv'
if os.path.exists(RESULTS_CSV):
    results = pd.read_csv(RESULTS_CSV)
    print(f'Total de experimentos: {len(results)}')
    print(f'\nTop 10 por F1-Score:')
    cols = ['experiment_id', 'learning_rate', 'weight_decay', 'accuracy', 'f1_score']
    if 'dropout_rate' in results.columns:
        cols.insert(3, 'dropout_rate')
    print(results.nlargest(10, 'f1_score')[cols].to_string(index=False))
else:
    print('No hay resultados. Verifica que los notebooks se ejecutaron correctamente.')

In [ ]:
# CELDA 10: Ver metricas del modelo base
import json

METRICS_PATH = 'experiments/cnrpark/base_model/metrics.json'
if os.path.exists(METRICS_PATH):
    with open(METRICS_PATH) as f:
        base_metrics = json.load(f)
    print('Metricas del Modelo Base:')
    for k, v in base_metrics.items():
        if isinstance(v, float):
            print(f'  {k}: {v*100:.2f}%')
        else:
            print(f'  {k}: {v}')
else:
    print('No se encontro metrics.json')

## Descargar Resultados

In [ ]:
# CELDA 11: Empaquetar y descargar resultados
import zipfile

with zipfile.ZipFile('smartparking_results.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk('experiments/cnrpark'):
        for file in files:
            filepath = os.path.join(root, file)
            zipf.write(filepath)
    for root, dirs, files in os.walk('reports/figures'):
        for file in files:
            filepath = os.path.join(root, file)
            zipf.write(filepath)

files.download('smartparking_results.zip')
print('Resultados descargados: smartparking_results.zip')